# 14j R50-IBN Feature Extraction

Kernel slug: `yahiaakhalafallah/14j-r50ibn-features`.

Extract FastReID SBS R50-IBN CityFlowV2 features from the same 929 Stage-1 tracklets used by 14h v3. Outputs are written as `embeddings_quaternary.npy`, `embedding_index.json`, and a compact validation summary for the downstream 4-way fusion sweep.

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile, re, random
from datetime import datetime
from pathlib import Path

if shutil.which("nvidia-smi"):
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0 and result.stdout.strip():
        gpu_name, compute_cap = result.stdout.strip().split(",", 1)
        match = re.search(r"(\d+)\.(\d+)", compute_cap)
        if match:
            major, minor = match.groups()
            sm = int(major) * 10 + int(minor)
            if sm < 70:
                print(f"GPU {gpu_name.strip()} sm_{sm}: installing torch 2.4.1+cu124")
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1+cu124", "torchvision==0.19.1+cu124",
                    "--index-url", "https://download.pytorch.org/whl/cu124",
                ])

import numpy as np
import cv2
from PIL import Image
import torch

SEED = 20260508
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"  GPU {index}: {torch.cuda.get_device_name(index)} ({props.total_memory / 1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Clone Repo And Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/pretrained-ensemble", REPO_URL, str(PROJECT)])
else:
    print("Repo already present; pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("filterpy", "ftfy", "lapx", "loguru", "omegaconf", "rich", "networkx>=3.1", "click", "motmetrics", "timm")
try:
    import torchreid
    print(f"torchreid ok: {getattr(torchreid, '__version__', 'unknown')}")
except ImportError:
    pip("git+https://github.com/KaiyangZhou/deep-person-reid.git")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=str(PROJECT))

FAILED = []
for label, module in [
    ("torch", "torch"),
    ("torchreid", "torchreid"),
    ("timm", "timm"),
    ("cv2", "cv2"),
    ("omegaconf", "omegaconf"),
]:
    try:
        __import__(module)
        print(f"  OK {label}")
    except ImportError as exc:
        print(f"  MISSING {label}: {exc}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED}")
print("Dependencies installed")

## 2. Resolve Inputs

In [ ]:
INPUT_ROOT = Path("/kaggle/input")


def find_input_dir(slug: str, owner_slug: str | None = None, hints=()) -> Path:
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct
    if owner_slug:
        owner, _, kernel = owner_slug.partition("/")
        nested = INPUT_ROOT / "notebooks" / owner / kernel
        if nested.exists():
            return nested
    lowered_slug = slug.lower()
    lowered_hints = tuple(str(hint).lower() for hint in hints)
    for path in list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []:
        if not path.is_dir():
            continue
        name = path.name.lower()
        if lowered_slug in name or (lowered_hints and all(hint in name for hint in lowered_hints)):
            return path
    return direct

CHECKPOINT_INPUT = find_input_dir(
    "09p-r50-ibn-cityflowv2-extended-checkpoint",
    hints=("09p", "r50", "ibn", "checkpoint"),
)
CHECKPOINT_PATH = CHECKPOINT_INPUT / "fastreid_r50_ibn_cityflowv2_extended_final.pth"
if not CHECKPOINT_PATH.exists():
    matches = sorted(INPUT_ROOT.rglob("fastreid_r50_ibn_cityflowv2_extended_final.pth")) if INPUT_ROOT.exists() else []
    if matches:
        CHECKPOINT_PATH = matches[0]
    else:
        raise FileNotFoundError(f"09p FastReID checkpoint not found under {CHECKPOINT_INPUT}")
print(f"09p checkpoint: {CHECKPOINT_PATH} ({CHECKPOINT_PATH.stat().st_size / 1024**2:.1f} MB)")

cityflow_candidates = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_INPUT = next((path for path in cityflow_candidates if path.exists()), None)
if CITYFLOW_INPUT is None:
    raise FileNotFoundError("CityFlowV2 dataset not found; attach thanhnguyenle/data-aicity-2023-track-2")
print(f"CityFlowV2 input: {CITYFLOW_INPUT}")

SOURCE_14H_DIR = find_input_dir(
    "14h-robust-tracklet-pooling",
    "yahiaakhalafallah/14h-robust-tracklet-pooling",
    hints=("14h", "robust", "pooling"),
)
SOURCE_10A_DIR = find_input_dir(
    "mtmc-10a-stages-0-2",
    "yahiaakhalafallah/mtmc-10a-stages-0-2",
    hints=("10a", "stages", "0", "2"),
)
print(f"14h source mount: {SOURCE_14H_DIR} exists={SOURCE_14H_DIR.exists()}")
print(f"10a source mount: {SOURCE_10A_DIR} exists={SOURCE_10A_DIR.exists()}")

## 3. Prepare CityFlowV2 Videos

In [ ]:
import re as regex

for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount:16s} {free / 1024**3:.1f} GB free / {total / 1024**3:.1f} GB total")

TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)
DATA_RAW_PARENT = PROJECT / "data" / "raw"
if not DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.exists():
        shutil.rmtree(DATA_RAW_PARENT)
    DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
    DATA_RAW_PARENT.symlink_to(TMP_DATA)

DATA_RAW = TMP_DATA / "cityflowv2"
DATA_RAW.mkdir(parents=True, exist_ok=True)
for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir():
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_name = f"{scene_dir.name}_{cam_dir.name}"
            flat_dir = DATA_RAW / flat_name
            if not flat_dir.exists():
                flat_dir.symlink_to(cam_dir)

cam_pattern = regex.compile(r"^S\d{2}_c\d{3}$")
cams = sorted(path.name for path in DATA_RAW.iterdir() if path.is_dir() and cam_pattern.match(path.name))
print(f"CityFlowV2 ready: {len(cams)} cameras")
for cam in cams:
    print(f"  {cam}")

## 4. Load Tracklets From 14h Or 10a

In [ ]:
def find_checkpoint_tar() -> Path:
    candidates = []
    for root in [SOURCE_14H_DIR, SOURCE_10A_DIR, INPUT_ROOT]:
        if root.exists():
            candidates.extend(sorted(root.rglob("checkpoint.tar.gz")))
    for path in candidates:
        text = str(path).lower()
        if "14h" in text or "10a" in text or "mtmc" in text:
            return path
    if candidates:
        return candidates[0]

    dl_dir = Path("/tmp/kaggle_10a_download")
    dl_dir.mkdir(parents=True, exist_ok=True)
    for candidate in ["yahiaakhalafallah/mtmc-10a-stages-0-2", "gumfreddy/mtmc-10a-stages-0-2"]:
        result = subprocess.run(
            ["kaggle", "kernels", "output", candidate, "--file-pattern", r"^checkpoint\.tar\.gz$", "-p", str(dl_dir)],
            capture_output=True,
            text=True,
        )
        print(result.stdout)
        print(result.stderr)
        checkpoint_path = dl_dir / "checkpoint.tar.gz"
        if checkpoint_path.exists() and checkpoint_path.stat().st_size > 0:
            print(f"10a checkpoint downloaded from {candidate}")
            return checkpoint_path
    raise FileNotFoundError("No checkpoint.tar.gz found in 14h/10a mounts or API fallback")

checkpoint_tar = find_checkpoint_tar()
EXTRACT_DIR = Path("/tmp/source_checkpoint")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {checkpoint_tar} ({checkpoint_tar.stat().st_size / 1024**2:.1f} MB)")
with tarfile.open(str(checkpoint_tar), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

metadata_path = EXTRACT_DIR / "run_metadata.json"
if metadata_path.exists():
    previous_meta = json.loads(metadata_path.read_text())
    PREV_RUN_NAME = previous_meta["run_name"]
    PREV_RUN_DIR = EXTRACT_DIR / PREV_RUN_NAME
else:
    stage1_dirs = sorted(EXTRACT_DIR.rglob("stage1"))
    if not stage1_dirs:
        raise FileNotFoundError(f"No stage1 directory found after extracting {checkpoint_tar}")
    PREV_RUN_DIR = stage1_dirs[0].parent
    PREV_RUN_NAME = PREV_RUN_DIR.name

print(f"Loaded source run: {PREV_RUN_NAME}")
for stage in ["stage1", "stage2"]:
    stage_dir = PREV_RUN_DIR / stage
    print(f"  {stage}: exists={stage_dir.exists()} files={len([p for p in stage_dir.rglob('*') if p.is_file()]) if stage_dir.exists() else 0}")

## 5. Extract R50-IBN Tracklet Features

In [ ]:
from datetime import datetime

import cv2
from PIL import Image

from src.core.io_utils import load_tracklets_by_camera
from src.stage2_features.embeddings import l2_normalize
from src.stage2_features.crop_extractor import CropExtractor
from src.training.datasets import build_test_transforms
from src.training.model import ReIDModelResNet50IBN

EXPECTED_TRACKLETS = 929
EXPECTED_FEATURE_DIM = 2048
RUN_NAME = f"run_14j_r50ibn_features_v4_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = Path("/kaggle/working/outputs/14j_v4_features")
STAGE1_DIR = RUN_DIR / "stage1"
FEATURE_DIR = RUN_DIR / "stage2"
PARTIAL_DIR = FEATURE_DIR / "per_camera"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)
if STAGE1_DIR.exists():
    shutil.rmtree(STAGE1_DIR)
shutil.copytree(PREV_RUN_DIR / "stage1", STAGE1_DIR)

tracklets_by_camera = load_tracklets_by_camera(STAGE1_DIR)
total_tracklets = sum(len(tracklets) for tracklets in tracklets_by_camera.values())
print(f"Tracklets: {total_tracklets} across {len(tracklets_by_camera)} cameras")
for camera_id in sorted(tracklets_by_camera):
    print(f"  {camera_id}: {len(tracklets_by_camera[camera_id])}")
if total_tracklets != EXPECTED_TRACKLETS:
    raise RuntimeError(f"Expected {EXPECTED_TRACKLETS} tracklets from 14h v3 source, found {total_tracklets}")

source_index_path = PREV_RUN_DIR / "stage2" / "embedding_index.json"
if not source_index_path.exists():
    raise FileNotFoundError(f"Missing 14h v3 embedding index: {source_index_path}")
source_index_map = json.loads(source_index_path.read_text(encoding="utf-8"))
if len(source_index_map) != EXPECTED_TRACKLETS:
    raise RuntimeError(f"Expected {EXPECTED_TRACKLETS} rows in 14h v3 embedding_index.json, found {len(source_index_map)}")

row_by_tracklet = {}
for row_index, record in enumerate(source_index_map):
    key = (str(record["camera_id"]), str(record["track_id"]))
    if key in row_by_tracklet:
        raise RuntimeError(f"Duplicate source embedding_index key: {key}")
    row_by_tracklet[key] = row_index

video_paths = {}
for cam_dir in sorted(DATA_RAW.glob("S*_c*")):
    video_path = cam_dir / "vdo.avi"
    if video_path.exists():
        video_paths[cam_dir.name] = str(video_path)
missing_videos = sorted(set(tracklets_by_camera) - set(video_paths))
if missing_videos:
    raise FileNotFoundError(f"Missing videos for cameras: {missing_videos}")


def unwrap_checkpoint_state_dict(checkpoint: object) -> dict:
    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            return checkpoint["state_dict"]
        if "model" in checkpoint:
            return checkpoint["model"]
        return checkpoint
    raise TypeError(f"Unsupported checkpoint type: {type(checkpoint)!r}")


class R50IBNFeatureExtractor:
    def __init__(self, checkpoint_path: Path, device: str):
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        state_dict = {
            key.replace("module.", "", 1): value
            for key, value in unwrap_checkpoint_state_dict(checkpoint).items()
        }
        if "classifier.weight" not in state_dict:
            raise KeyError("Expected classifier.weight in 09p R50-IBN checkpoint")
        num_classes = int(state_dict["classifier.weight"].shape[0])
        self.model = ReIDModelResNet50IBN(
            num_classes=num_classes,
            last_stride=1,
            pretrained=False,
            gem_p=3.0,
            eval_feature="global",
        )
        missing, unexpected = self.model.load_state_dict(state_dict, strict=True)
        if missing or unexpected:
            raise RuntimeError(f"Strict checkpoint load mismatch: missing={missing}, unexpected={unexpected}")
        self.model = self.model.to(device).eval()
        self.device = device
        self.transform = build_test_transforms(height=256, width=256)
        self.feature_dim = int(self.model.feat_dim)
        self.num_classes = num_classes
        print(json.dumps({
            "checkpoint_path": str(checkpoint_path),
            "strict_load": True,
            "num_classes": num_classes,
            "feature_dim": self.feature_dim,
            "eval_feature": self.model.eval_feature,
            "preprocess": "src.training.datasets.build_test_transforms(height=256,width=256)",
            "flip_eval": True,
        }, indent=2))

    def _preprocess(self, crops: list[np.ndarray]) -> torch.Tensor:
        tensors = []
        for crop in crops:
            rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            tensors.append(self.transform(Image.fromarray(rgb)))
        return torch.stack(tensors, dim=0)

    @torch.no_grad()
    def extract_features(self, crops: list[np.ndarray], batch_size: int = 64) -> np.ndarray:
        if not crops:
            return np.empty((0, self.feature_dim), dtype=np.float32)
        all_features = []
        for start_idx in range(0, len(crops), batch_size):
            batch = crops[start_idx:start_idx + batch_size]
            images = self._preprocess(batch).to(self.device)
            features = self.model(images)
            flipped_features = self.model(torch.flip(images, dims=[3]))
            features = (features + flipped_features) / 2.0
            features = torch.nn.functional.normalize(features, p=2, dim=1)
            all_features.append(features.float().cpu().numpy())
        return np.concatenate(all_features, axis=0).astype(np.float32)

    def get_tracklet_embedding_from_scored_crops(
        self,
        scored_crops,
        quality_temperature: float = 3.0,
    ) -> np.ndarray | None:
        if not scored_crops:
            return None
        crop_features = self.extract_features([sc.image for sc in scored_crops])
        if crop_features.shape[0] == 0:
            return None
        qualities = np.array([sc.quality for sc in scored_crops], dtype=np.float32)
        logits = qualities * float(quality_temperature)
        logits = logits - logits.max()
        weights = np.exp(logits).astype(np.float32)
        weights = weights / max(float(weights.sum()), 1e-8)
        pooled = (crop_features * weights[:, np.newaxis]).sum(axis=0).astype(np.float32)
        return l2_normalize(pooled[np.newaxis, :])[0]


def make_drop_record(camera_id: str, tracklet, row_index: int, reason: str) -> dict:
    frames = [tf.frame_id for tf in tracklet.frames]
    areas = [float((tf.bbox[2] - tf.bbox[0]) * (tf.bbox[3] - tf.bbox[1])) for tf in tracklet.frames]
    confidences = [float(tf.confidence) for tf in tracklet.frames]
    return {
        "index": int(row_index),
        "camera_id": camera_id,
        "track_id": int(tracklet.track_id),
        "class_id": int(tracklet.class_id),
        "reason": reason,
        "frames": int(len(tracklet.frames)),
        "first_frame_id": int(min(frames)) if frames else None,
        "last_frame_id": int(max(frames)) if frames else None,
        "min_bbox_area": float(min(areas)) if areas else None,
        "max_bbox_area": float(max(areas)) if areas else None,
        "min_confidence": float(min(confidences)) if confidences else None,
        "max_confidence": float(max(confidences)) if confidences else None,
    }


def write_json(path: Path, payload: object) -> None:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def persist_camera_outputs(
    camera_id: str,
    row_indices: list[int],
    records: list[dict],
    camera_matrix: np.ndarray,
    camera_drops: list[dict],
    elapsed_seconds: float,
) -> None:
    camera_prefix = PARTIAL_DIR / camera_id
    np.save(camera_prefix.with_name(f"{camera_id}_embeddings_quaternary.npy"), camera_matrix.astype(np.float32))
    np.save(camera_prefix.with_name(f"{camera_id}_row_indices.npy"), np.array(row_indices, dtype=np.int64))
    write_json(camera_prefix.with_name(f"{camera_id}_embedding_index.json"), records)
    write_json(camera_prefix.with_name(f"{camera_id}_dropped_tracklets.json"), camera_drops)
    status = {
        "camera_id": camera_id,
        "rows": int(len(row_indices)),
        "extracted": int(len(row_indices) - len(camera_drops)),
        "dropped": int(len(camera_drops)),
        "elapsed_minutes": round(elapsed_seconds / 60.0, 2),
        "embedding_shape": list(camera_matrix.shape),
        "row_indices_path": str(camera_prefix.with_name(f"{camera_id}_row_indices.npy")),
        "embedding_path": str(camera_prefix.with_name(f"{camera_id}_embeddings_quaternary.npy")),
        "embedding_index_path": str(camera_prefix.with_name(f"{camera_id}_embedding_index.json")),
        "dropped_tracklets_path": str(camera_prefix.with_name(f"{camera_id}_dropped_tracklets.json")),
    }
    write_json(camera_prefix.with_name(f"{camera_id}_status.json"), status)


crop_extractor = CropExtractor(
    samples_per_tracklet=48,
    min_area=32 * 32,
    min_quality=0.05,
)
r50_reid = R50IBNFeatureExtractor(CHECKPOINT_PATH, DEVICE)
if r50_reid.feature_dim != EXPECTED_FEATURE_DIM:
    raise RuntimeError(f"Expected feature_dim={EXPECTED_FEATURE_DIM}, got {r50_reid.feature_dim}")

embedding_matrix = np.zeros((len(source_index_map), EXPECTED_FEATURE_DIM), dtype=np.float32)
filled_mask = np.zeros((len(source_index_map),), dtype=bool)
processed_mask = np.zeros((len(source_index_map),), dtype=bool)
per_camera = {}
dropped_by_camera = {}
dropped_tracklets = []
start = time.time()

for camera_id in sorted(tracklets_by_camera):
    camera_start = time.time()
    tracklets = tracklets_by_camera[camera_id]
    video_path = video_paths[camera_id]
    camera_row_indices = []
    camera_records = []
    camera_drops = []
    camera_embeddings = 0
    print(f"Extracting {camera_id}: {len(tracklets)} tracklets")
    for tracklet in tracklets:
        key = (str(camera_id), str(tracklet.track_id))
        if key not in row_by_tracklet:
            raise RuntimeError(f"Tracklet missing from 14h v3 embedding index: {key}")
        row_index = row_by_tracklet[key]
        source_record = dict(source_index_map[row_index])
        camera_row_indices.append(row_index)
        camera_records.append(source_record)
        processed_mask[row_index] = True

        scored_crops = crop_extractor.extract_crops(tracklet, video_path)
        if not scored_crops:
            drop_record = make_drop_record(camera_id, tracklet, row_index, "no_crops_survived_filtering")
            camera_drops.append(drop_record)
            dropped_tracklets.append(drop_record)
            print(
                f"WARNING: {camera_id} tracklet {tracklet.track_id} row {row_index}: "
                "0 crops survived filtering; writing zero-vector placeholder"
            )
            continue

        embedding = r50_reid.get_tracklet_embedding_from_scored_crops(scored_crops, quality_temperature=3.0)
        if embedding is None:
            drop_record = make_drop_record(camera_id, tracklet, row_index, "no_embedding_from_crops")
            camera_drops.append(drop_record)
            dropped_tracklets.append(drop_record)
            print(
                f"WARNING: {camera_id} tracklet {tracklet.track_id} row {row_index}: "
                "model returned no embedding; writing zero-vector placeholder"
            )
            continue

        embedding_matrix[row_index] = embedding.astype(np.float32)
        filled_mask[row_index] = True
        camera_embeddings += 1

    camera_elapsed = time.time() - camera_start
    camera_matrix = embedding_matrix[np.array(camera_row_indices, dtype=np.int64)]
    persist_camera_outputs(
        camera_id=camera_id,
        row_indices=camera_row_indices,
        records=camera_records,
        camera_matrix=camera_matrix,
        camera_drops=camera_drops,
        elapsed_seconds=camera_elapsed,
    )
    per_camera[camera_id] = camera_embeddings
    dropped_by_camera[camera_id] = len(camera_drops)
    partial_status = {
        "run_name": RUN_NAME,
        "version": "v4",
        "completed_cameras": sorted(per_camera),
        "per_camera": per_camera,
        "dropped_by_camera": dropped_by_camera,
        "filled_rows": int(filled_mask.sum()),
        "processed_rows": int(processed_mask.sum()),
        "dropped_rows": int(len(dropped_tracklets)),
        "expected_rows": int(EXPECTED_TRACKLETS),
    }
    write_json(FEATURE_DIR / "partial_status.json", partial_status)
    print(
        f"  {camera_id}: extracted={camera_embeddings} dropped={len(camera_drops)} "
        f"saved_partial={PARTIAL_DIR / (camera_id + '_embeddings_quaternary.npy')} "
        f"elapsed={camera_elapsed / 60:.1f} min"
    )

elapsed = time.time() - start
if not processed_mask.all():
    missing_rows = np.flatnonzero(~processed_mask).astype(int).tolist()
    raise RuntimeError(f"Rows present in 14h v3 embedding index were not visited: {missing_rows[:20]}")
if int(filled_mask.sum()) + len(dropped_tracklets) != EXPECTED_TRACKLETS:
    raise RuntimeError(
        f"Internal accounting mismatch: filled={int(filled_mask.sum())} "
        f"dropped={len(dropped_tracklets)} expected={EXPECTED_TRACKLETS}"
    )

embedding_matrix = l2_normalize(embedding_matrix).astype(np.float32)
if embedding_matrix.shape != (EXPECTED_TRACKLETS, EXPECTED_FEATURE_DIM):
    raise RuntimeError(f"Unexpected embedding matrix shape: {embedding_matrix.shape}")
if not np.isfinite(embedding_matrix).all():
    raise RuntimeError("R50-IBN embeddings contain NaN/Inf")

dropped_indices = [
    {
        "index": int(record["index"]),
        "camera_id": record["camera_id"],
        "track_id": int(record["track_id"]),
        "reason": record["reason"],
    }
    for record in dropped_tracklets
]
dropped_indices = sorted(dropped_indices, key=lambda item: item["index"])
dropped_rows = [int(item["index"]) for item in dropped_indices]
if dropped_rows:
    print(f"WARNING: zero-filled {len(dropped_rows)} dropped tracklets at rows {dropped_rows}")
else:
    print("No dropped tracklets; all rows contain R50-IBN embeddings")

np.save(FEATURE_DIR / "embeddings_quaternary.npy", embedding_matrix)
np.save(FEATURE_DIR / "embeddings_secondary.npy", embedding_matrix)
write_json(FEATURE_DIR / "embedding_index.json", source_index_map)
write_json(FEATURE_DIR / "dropped_tracklets.json", sorted(dropped_tracklets, key=lambda item: item["index"]))
write_json(FEATURE_DIR / "dropped_indices.json", {
    "count": int(len(dropped_indices)),
    "fill_value": 0.0,
    "indices": dropped_rows,
    "tracklets": dropped_indices,
})

norms = np.linalg.norm(embedding_matrix, axis=1)
non_dropped_mask = np.ones((EXPECTED_TRACKLETS,), dtype=bool)
if dropped_rows:
    non_dropped_mask[np.array(dropped_rows, dtype=np.int64)] = False
non_dropped_norms = norms[non_dropped_mask]
dropped_norms = norms[~non_dropped_mask]
summary = {
    "experiment": "14j-r50ibn-features",
    "kernel": "yahiaakhalafallah/14j-r50-ibn-features",
    "version": "v4",
    "run_name": RUN_NAME,
    "source_run_name": PREV_RUN_NAME,
    "checkpoint_dataset": "gumfreddy/09p-r50-ibn-cityflowv2-extended-checkpoint",
    "checkpoint_path": str(CHECKPOINT_PATH),
    "checkpoint_strict_load": True,
    "source_14h_mount": str(SOURCE_14H_DIR),
    "source_10a_mount": str(SOURCE_10A_DIR),
    "source_embedding_index_path": str(source_index_path),
    "embedding_path": str(FEATURE_DIR / "embeddings_quaternary.npy"),
    "secondary_alias_path": str(FEATURE_DIR / "embeddings_secondary.npy"),
    "embedding_index_path": str(FEATURE_DIR / "embedding_index.json"),
    "dropped_tracklets_path": str(FEATURE_DIR / "dropped_tracklets.json"),
    "dropped_indices_path": str(FEATURE_DIR / "dropped_indices.json"),
    "per_camera_partial_dir": str(PARTIAL_DIR),
    "shape": list(embedding_matrix.shape),
    "feature_dim": int(embedding_matrix.shape[1]),
    "dtype": str(embedding_matrix.dtype),
    "per_camera": per_camera,
    "dropped_by_camera": dropped_by_camera,
    "dropped_tracklets": sorted(dropped_tracklets, key=lambda item: item["index"]),
    "dropped_indices": dropped_rows,
    "norm_min": float(norms.min()),
    "norm_max": float(norms.max()),
    "norm_mean": float(norms.mean()),
    "non_dropped_norm_min": float(non_dropped_norms.min()) if non_dropped_norms.size else None,
    "non_dropped_norm_max": float(non_dropped_norms.max()) if non_dropped_norms.size else None,
    "dropped_norm_min": float(dropped_norms.min()) if dropped_norms.size else None,
    "dropped_norm_max": float(dropped_norms.max()) if dropped_norms.size else None,
    "has_nan": bool(np.isnan(embedding_matrix).any()),
    "has_inf": bool(np.isinf(embedding_matrix).any()),
    "aggregation": "softmax_quality_mean_temperature_3_original_plus_hflip_zero_fill_dropped",
    "samples_per_tracklet": 48,
    "crop_min_area": 32 * 32,
    "dropout_handling": "zero_vector_placeholder_preserve_14h_embedding_index_order",
    "input_size": [256, 256],
    "preprocess": "build_test_transforms_09p_eval_imagenet_bicubic",
    "elapsed_minutes": round(elapsed / 60.0, 2),
}
summary_path = RUN_DIR / "14j_features_summary.json"
write_json(summary_path, summary)
write_json(Path("/kaggle/working/14j_features_summary.json"), summary)

print(json.dumps(summary, indent=2))

## 6. Validate Output Contract

In [ ]:
embedding_matrix = np.load(FEATURE_DIR / "embeddings_quaternary.npy")
alias_matrix = np.load(FEATURE_DIR / "embeddings_secondary.npy")
index_map = json.loads((FEATURE_DIR / "embedding_index.json").read_text(encoding="utf-8"))
dropped_payload = json.loads((FEATURE_DIR / "dropped_indices.json").read_text(encoding="utf-8"))
dropped_indices = [int(index) for index in dropped_payload["indices"]]

assert embedding_matrix.shape == (929, 2048), embedding_matrix.shape
assert alias_matrix.shape == embedding_matrix.shape, alias_matrix.shape
assert len(index_map) == 929, len(index_map)
assert np.isfinite(embedding_matrix).all()
assert np.array_equal(embedding_matrix, alias_matrix)

norms = np.linalg.norm(embedding_matrix, axis=1)
non_dropped_mask = np.ones((embedding_matrix.shape[0],), dtype=bool)
if dropped_indices:
    non_dropped_mask[np.array(dropped_indices, dtype=np.int64)] = False
    assert np.allclose(embedding_matrix[np.array(dropped_indices, dtype=np.int64)], 0.0)
    assert np.allclose(norms[~non_dropped_mask], 0.0)
assert float(np.max(np.abs(norms[non_dropped_mask] - 1.0))) < 1e-4, (float(norms[non_dropped_mask].min()), float(norms[non_dropped_mask].max()))

print(json.dumps({
    "validated": True,
    "shape": list(embedding_matrix.shape),
    "index_rows": len(index_map),
    "dropped_count": len(dropped_indices),
    "dropped_indices": dropped_indices,
    "norm_min": float(norms.min()),
    "norm_max": float(norms.max()),
    "non_dropped_norm_min": float(norms[non_dropped_mask].min()),
    "non_dropped_norm_max": float(norms[non_dropped_mask].max()),
}, indent=2))